In [1]:
!pip install geopandas

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ALL TIME

In [7]:
# Semarang
import geopandas as gpd
import pandas as pd
import glob
import os

def aggregate_traffic_data(folder_path, traffic_column):
    """
    Aggregate traffic data from multiple GPKG files and save as GPKG.

    Parameters:
    folder_path (str): Path to the folder containing GPKG files
    traffic_column (str): Name of the column containing traffic data

    Returns:
    geopandas.GeoDataFrame: Aggregated traffic data by segment_id with geometry
    """
    # Get list of all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(folder_path, "*.gpkg"))

    # Check if any files were found
    if not gpkg_files:
        print(f"No GPKG files found in {folder_path}")
        return None


    # List to store DataFrames for statistics
    all_data = []

    # Read the first file to get geometries
    try:
        first_gdf = gpd.read_file(gpkg_files[0])
        # Keep segment_id and geometry for final result
        base_geometry = first_gdf[['segment_id', 'geometry']].copy()
    except KeyError:
        print(f"Error: The first file {gpkg_files[0]} does not contain 'segment_id' or 'geometry' columns.")
        return None
    except Exception as e:
        print(f"Error reading the first file {gpkg_files[0]}: {str(e)}")
        return None


    # Read each GPKG file
    for file in gpkg_files:
        try:
            # Read the GPKG file
            gdf = gpd.read_file(file)

            # Check if required columns exist
            if 'segment_id' not in gdf.columns or traffic_column not in gdf.columns:
                print(f"Skipping file {file}: Missing 'segment_id' or '{traffic_column}' column.")
                continue

            # Extract relevant columns (segment_id and traffic column)
            df = gdf[['segment_id', traffic_column]].copy()

            # Add timestamp (from filename if available, or file modification time)
            timestamp = os.path.getmtime(file)  # Modify based on your filename pattern
            df['timestamp'] = pd.to_datetime(timestamp, unit='s')

            all_data.append(df)

        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

    # Combine all DataFrames
    if not all_data:
        print("No valid data found in any of the GPKG files.")
        return None

    combined_data = pd.concat(all_data, ignore_index=True)

    # Perform aggregations
    aggregations = {
        traffic_column: ['mean', 'min', 'max', 'std', 'count'],
        'timestamp': ['min', 'max']  # First and last observation
    }

    aggregated_data = combined_data.groupby('segment_id').agg(aggregations)

    # Flatten column names
    aggregated_data.columns = [f"{col[0]}_{col[1]}" for col in aggregated_data.columns]

    # Reset index to make segment_id a column
    aggregated_data = aggregated_data.reset_index()

    # Merge aggregated statistics with geometry
    final_gdf = base_geometry.merge(aggregated_data, on='segment_id', how='left')

    return final_gdf

# Example usage
if __name__ == "__main__":
    # Set your folder path and traffic column name
    folder_path = "/content/traffic-data/traffic_data_smg"
    traffic_column = "jam_factor"
    output_file = "/content/traffic-data/smg_aggregated_traffic_all.gpkg"

    # Aggregate the data
    result_gdf = aggregate_traffic_data(folder_path, traffic_column)

    # Save as GPKG
    if result_gdf is not None:
        result_gdf.to_file(output_file, driver='GPKG')

        # Print some statistics
        print("\nAggregation Summary:")
        print(f"Total segments: {len(result_gdf)}")
        print("\nSample of aggregated data:")
        print(result_gdf.head())

Error: The first file /content/drive/Othercomputers/MyMac/Documents/Micro-mobility/traffic-data/traffic_data_smg/semarang_traffic_20250619_090001.gpkg does not contain 'segment_id' or 'geometry' columns.


In [ ]:
# Bandung
import geopandas as gpd
import pandas as pd
import glob
import os

def aggregate_traffic_data(folder_path, traffic_column):
    """
    Aggregate traffic data from multiple GPKG files and save as GPKG.

    Parameters:
    folder_path (str): Path to the folder containing GPKG files
    traffic_column (str): Name of the column containing traffic data

    Returns:
    geopandas.GeoDataFrame: Aggregated traffic data by segment_id with geometry
    """
    # Get list of all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(folder_path, "*.gpkg"))

    # List to store DataFrames for statistics
    all_data = []

    # Read the first file to get geometries
    first_gdf = gpd.read_file(gpkg_files[0])
    # Keep segment_id and geometry for final result
    base_geometry = first_gdf[['segment_id', 'geometry']].copy()

    # Read each GPKG file
    for file in gpkg_files:
        try:
            # Read the GPKG file
            gdf = gpd.read_file(file)

            # Extract relevant columns (segment_id and traffic column)
            df = gdf[['segment_id', traffic_column]].copy()

            # Add timestamp (from filename if available, or file modification time)
            timestamp = os.path.getmtime(file)  # Modify based on your filename pattern
            df['timestamp'] = pd.to_datetime(timestamp, unit='s')

            all_data.append(df)

        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

    # Combine all DataFrames
    combined_data = pd.concat(all_data, ignore_index=True)

    # Perform aggregations
    aggregations = {
        traffic_column: ['mean', 'min', 'max', 'std', 'count'],
        'timestamp': ['min', 'max']  # First and last observation
    }

    aggregated_data = combined_data.groupby('segment_id').agg(aggregations)

    # Flatten column names
    aggregated_data.columns = [f"{col[0]}_{col[1]}" for col in aggregated_data.columns]

    # Reset index to make segment_id a column
    aggregated_data = aggregated_data.reset_index()

    # Merge aggregated statistics with geometry
    final_gdf = base_geometry.merge(aggregated_data, on='segment_id', how='left')

    return final_gdf

# Example usage
if __name__ == "__main__":
    # Set your folder path and traffic column name
    folder_path = "traffic_data_bdg"
    traffic_column = "jam_factor"
    output_file = "bdg_aggregated_traffic_all.gpkg"

    # Aggregate the data
    result_gdf = aggregate_traffic_data(folder_path, traffic_column)

    # Save as GPKG
    result_gdf.to_file(output_file, driver='GPKG')

    # Print some statistics
    print("\nAggregation Summary:")
    print(f"Total segments: {len(result_gdf)}")
    print("\nSample of aggregated data:")
    print(result_gdf.head())


Aggregation Summary:
Total segments: 3066

Sample of aggregated data:
                         segment_id  \
0  4262c5f5e929b808f8a737638bfdc3db   
1  079e7f1523ffeb363a5d03294e368f39   
2  e6a4938b8696363b32e4a52bdb8a3d00   
3  3c5546b0fde98b31b0379848ec2558e7   
4  c32c068cf953569147f575e0639908ed   

                                            geometry  jam_factor_mean  \
0  MULTILINESTRING ((107.66160 -6.95912, 107.6619...         0.352994   
1  MULTILINESTRING ((107.50230 -6.89813, 107.5024...         0.249257   
2  MULTILINESTRING ((107.53819 -6.91907, 107.5381...         0.533658   
3  MULTILINESTRING ((107.51489 -7.02702, 107.5149...         0.243524   
4  MULTILINESTRING ((107.64422 -6.91972, 107.6442...         1.351486   

   jam_factor_min  jam_factor_max  jam_factor_std  jam_factor_count  \
0             0.0             4.3        0.540471              4308   
1             0.0             8.2        0.863283              4308   
2             0.0             7.3        0

In [ ]:
# Jakarta
import geopandas as gpd
import pandas as pd
import glob
import os

def aggregate_traffic_data(folder_path, traffic_column):
    """
    Aggregate traffic data from multiple GPKG files and save as GPKG.

    Parameters:
    folder_path (str): Path to the folder containing GPKG files
    traffic_column (str): Name of the column containing traffic data

    Returns:
    geopandas.GeoDataFrame: Aggregated traffic data by segment_id with geometry
    """
    # Get list of all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(folder_path, "*.gpkg"))

    # List to store DataFrames for statistics
    all_data = []

    # Read the first file to get geometries
    first_gdf = gpd.read_file(gpkg_files[0])
    # Keep segment_id and geometry for final result
    base_geometry = first_gdf[['segment_id', 'geometry']].copy()

    # Read each GPKG file
    for file in gpkg_files:
        try:
            # Read the GPKG file
            gdf = gpd.read_file(file)

            # Extract relevant columns (segment_id and traffic column)
            df = gdf[['segment_id', traffic_column]].copy()

            # Add timestamp (from filename if available, or file modification time)
            timestamp = os.path.getmtime(file)  # Modify based on your filename pattern
            df['timestamp'] = pd.to_datetime(timestamp, unit='s')

            all_data.append(df)

        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

    # Combine all DataFrames
    combined_data = pd.concat(all_data, ignore_index=True)

    # Perform aggregations
    aggregations = {
        traffic_column: ['mean', 'min', 'max', 'std', 'count'],
        'timestamp': ['min', 'max']  # First and last observation
    }

    aggregated_data = combined_data.groupby('segment_id').agg(aggregations)

    # Flatten column names
    aggregated_data.columns = [f"{col[0]}_{col[1]}" for col in aggregated_data.columns]

    # Reset index to make segment_id a column
    aggregated_data = aggregated_data.reset_index()

    # Merge aggregated statistics with geometry
    final_gdf = base_geometry.merge(aggregated_data, on='segment_id', how='left')

    return final_gdf

# Example usage
if __name__ == "__main__":
    # Set your folder path and traffic column name
    folder_path = "traffic_data_jkt"
    traffic_column = "jam_factor"
    output_file = "jkt_aggregated_traffic_all.gpkg"

    # Aggregate the data
    result_gdf = aggregate_traffic_data(folder_path, traffic_column)

    # Save as GPKG
    result_gdf.to_file(output_file, driver='GPKG')

    # Print some statistics
    print("\nAggregation Summary:")
    print(f"Total segments: {len(result_gdf)}")
    print("\nSample of aggregated data:")
    print(result_gdf.head())

Error reading traffic_data_jkt/jakarta_traffic_20241209_011838.gpkg: "['segment_id'] not in index"

Aggregation Summary:
Total segments: 14651

Sample of aggregated data:
                         segment_id  \
0  9e5aadc6a5de7a1b2835f3ad6a610036   
1  0d3ede8200880480c13bb76c851da3b2   
2  fef8cc5810ced180a914703b61a4649d   
3  8cebfc230d5fc5a279ad2b010b56f51e   
4  e2d7b22e8fbfa6635e6e7ab1a6e241ae   

                                            geometry  jam_factor_mean  \
0  MULTILINESTRING ((107.06135 -6.12474, 107.0613...         0.384071   
1  MULTILINESTRING ((106.98778 -6.17554, 106.9878...         0.681428   
2  MULTILINESTRING ((106.71669 -6.17855, 106.7167...         0.054115   
3  MULTILINESTRING ((106.70694 -6.17206, 106.7063...         0.209506   
4  MULTILINESTRING ((106.80247 -6.16592, 106.8026...         1.128658   

   jam_factor_min  jam_factor_max  jam_factor_std  jam_factor_count  \
0             0.0             1.5        0.337614              4313   
1            

In [ ]:
# # Semarang

# import geopandas as gpd
# import pandas as pd
# import numpy as np
# import glob
# import os
# from datetime import datetime

# def extract_timestamp_from_filename(filename):
#     """
#     Extract timestamp from filename pattern: city_traffic_YYYYMMDD_HHMMSS.gpkg
#     """
#     try:
#         # Get just the filename without path and remove extension
#         base_name = os.path.splitext(os.path.basename(filename))[0]
#         # Split by underscore and get date and time parts
#         parts = base_name.split('_')
#         date_str = parts[2]
#         time_str = parts[3]
#         # Combine date and time and parse
#         datetime_str = f"{date_str}_{time_str}"
#         return datetime.strptime(datetime_str, "%Y%m%d_%H%M%S")
#     except Exception as e:
#         print(f"Error parsing timestamp from {filename}: {str(e)}")
#         return None

# def get_time_period(hour):
#     """
#     Define time periods in a day
#     """
#     if 0 <= hour < 6:
#         return 'Night (00:00-06:00)'
#     elif 6 <= hour < 9:
#         return 'Morning Peak (06:00-09:00)'
#     elif 9 <= hour < 12:
#         return 'Morning Off-peak (09:00-12:00)'
#     elif 12 <= hour < 14:
#         return 'Lunch Hours (12:00-14:00)'
#     elif 14 <= hour < 16:
#         return 'Afternoon Off-peak (14:00-16:00)'
#     elif 16 <= hour < 19:
#         return 'Evening Peak (16:00-19:00)'
#     elif 19 <= hour < 22:
#         return 'Evening Off-peak (19:00-22:00)'
#     else:
#         return 'Late Night (22:00-00:00)'

# def analyze_city_traffic(city_folder, traffic_column):
#     """
#     Analyze traffic patterns for a single city

#     Parameters:
#     city_folder (str): Path to the folder containing city's GPKG files
#     traffic_column (str): Name of the column containing traffic data
#     """
#     # Get city name from folder name
#     city = os.path.basename(city_folder).split('_')[-1]  # Gets 'smg' from 'traffic_data_smg'

#     # Get all GPKG files in the folder
#     gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))
#     all_data = []

#     for file in gpkg_files:
#         try:
#             gdf = gpd.read_file(file)
#             df = gdf[['segment_id', traffic_column]].copy()

#             # Extract timestamp from filename
#             timestamp = extract_timestamp_from_filename(file)
#             if timestamp:
#                 df['timestamp'] = timestamp
#                 all_data.append(df)

#         except Exception as e:
#             print(f"Error reading {file}: {str(e)}")

#     if not all_data:
#         print(f"No valid data found in {city_folder}")
#         return None

#     # Combine all data
#     combined_data = pd.concat(all_data, ignore_index=True)

#     # Extract time components
#     combined_data['hour'] = combined_data['timestamp'].dt.hour
#     combined_data['weekday'] = combined_data['timestamp'].dt.day_name()
#     combined_data['time_period'] = combined_data['hour'].apply(get_time_period)

#     # Analyze by time period
#     time_stats = combined_data.groupby('time_period').agg({
#         traffic_column: ['mean', 'std', 'count'],
#         'segment_id': 'nunique'
#     }).round(2)

#     # Analyze by weekday
#     weekday_stats = combined_data.groupby('weekday').agg({
#         traffic_column: ['mean', 'std', 'count']
#     }).round(2)

#     # Find peak hours
#     hourly_avg = combined_data.groupby('hour')[traffic_column].mean()
#     worst_hour = hourly_avg.idxmax()
#     best_hour = hourly_avg.idxmin()

#     return {
#         'city': city,
#         'combined_data': combined_data,
#         'time_stats': time_stats,
#         'weekday_stats': weekday_stats,
#         'worst_hour': worst_hour,
#         'best_hour': best_hour,
#         'hourly_avg': hourly_avg
#     }

# def plot_daily_pattern(analysis_result):
#     """
#     Create hourly traffic pattern plot for the city
#     """
#     import matplotlib.pyplot as plt

#     plt.figure(figsize=(15, 6))

#     hourly_avg = analysis_result['hourly_avg']
#     city = analysis_result['city']

#     plt.plot(hourly_avg.index, hourly_avg.values, marker='o')
#     plt.xlabel('Hour of Day')
#     plt.ylabel('Average Traffic Condition')
#     plt.title(f'Daily Traffic Pattern - {city.upper()}')
#     plt.grid(True)
#     plt.savefig(f'daily_traffic_pattern_{city}.png')
#     plt.close()

# # Example usage
# if __name__ == "__main__":
#     city_folder = "traffic_data_smg"  # Change this to your folder path
#     traffic_column = "jam_factor"

#     # Analyze patterns
#     analysis_result = analyze_city_traffic(city_folder, traffic_column)

#     if analysis_result:
#         # Print results
#         print(f"\nTraffic Analysis for {analysis_result['city'].upper()}")

#         print("\nTraffic Analysis by Time Period:")
#         print(analysis_result['time_stats'])

#         print("\nTraffic Analysis by Weekday:")
#         print(analysis_result['weekday_stats'])

#         print(f"\nPeak Traffic Hour: {analysis_result['worst_hour']:02d}:00")
#         print(f"Best Traffic Hour: {analysis_result['best_hour']:02d}:00")

#         # Create visualization
#         plot_daily_pattern(analysis_result)

KeyboardInterrupt: 

In [10]:
import geopandas as gpd
import glob
import os

def check_gpkg_structure(city_folder):
    """
    Check the structure of GPKG files to identify actual column names
    """
    # Get list of all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))

    if not gpkg_files:
        print(f"No GPKG files found in {city_folder}")
        return

    print(f"Found {len(gpkg_files)} GPKG files")

    # Check the first few files
    for i, file in enumerate(gpkg_files[:3]):  # Check first 3 files
        print(f"\n--- File {i+1}: {os.path.basename(file)} ---")
        try:
            gdf = gpd.read_file(file)
            print(f"Shape: {gdf.shape}")
            print(f"Columns: {list(gdf.columns)}")
            print(f"Geometry column: {gdf.geometry.name}")

            # Show first few rows to understand the data structure
            print("\nFirst 3 rows:")
            print(gdf.head(3))

        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

# Usage
city_folder = "/content/traffic-data/traffic_data_smg"  # Change this to your folder path
check_gpkg_structure(city_folder)

Found 7699 GPKG files

--- File 1: semarang_traffic_20250515_190001.gpkg ---
Shape: (1080, 9)
Columns: ['id', 'speed', 'speed_uncapped', 'free_flow', 'jam_factor', 'confidence', 'traversability', 'timestamp', 'geometry']
Geometry column: geometry

First 3 rows:
    id     speed  speed_uncapped  free_flow  jam_factor  confidence  \
0  1.0  8.333334        8.333334  11.111112         3.7        0.70   
1  1.0  5.833334        5.833334   9.722222         2.8        0.99   
2  1.0  3.611111        3.611111   3.888889         0.0        0.84   

  traversability                        timestamp  \
0           open 2025-05-15 12:00:05.939000+00:00   
1           open 2025-05-15 12:00:05.939000+00:00   
2           open 2025-05-15 12:00:05.939000+00:00   

                                            geometry  
0  MULTILINESTRING ((110.52336 -6.937, 110.52213 ...  
1  MULTILINESTRING ((110.43395 -6.96966, 110.4339...  
2  MULTILINESTRING ((110.42652 -6.97833, 110.4265...  

--- File 2: semaran

In [11]:
# Semarang aggregation by period

# Semarang aggregation by period - FIXED VERSION

import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime

def extract_timestamp_from_filename(filename):
    """
    Extract timestamp from filename pattern: city_traffic_YYYYMMDD_HHMMSS.gpkg
    """
    try:
        # Get just the filename without path and remove extension
        base_name = os.path.splitext(os.path.basename(filename))[0]
        # Split by underscore and get date and time parts
        parts = base_name.split('_')
        date_str = parts[2]
        time_str = parts[3]
        # Combine date and time and parse
        datetime_str = f"{date_str}_{time_str}"
        return datetime.strptime(datetime_str, "%Y%m%d_%H%M%S")
    except Exception as e:
        print(f"Error parsing timestamp from {filename}: {str(e)}")
        return None

def get_time_period(hour):
    """
    Define time periods in a day
    """
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 9:
        return 'morning_peak'
    elif 9 <= hour < 12:
        return 'morning_offpeak'
    elif 12 <= hour < 14:
        return 'lunch_hours'
    elif 14 <= hour < 16:
        return 'afternoon_offpeak'
    elif 16 <= hour < 19:
        return 'evening_peak'
    elif 19 <= hour < 22:
        return 'evening_offpeak'
    else:
        return 'late_night'

def analyze_city_traffic_by_period(city_folder, traffic_column, output_folder):
    """
    Analyze traffic patterns and save GPKG files for each time period

    Parameters:
    city_folder (str): Path to the folder containing city's GPKG files
    traffic_column (str): Name of the column containing traffic data
    output_folder (str): Path to save the output GPKG files
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get city name from folder name
    city = os.path.basename(city_folder).split('_')[-1]  # Gets 'smg' from 'traffic_data_smg'

    # Get list of all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))

    # Check if any files were found
    if not gpkg_files:
        print(f"No GPKG files found in {city_folder}")
        return

    # Get first file to get the geometry
    try:
        first_file = gpkg_files[0]
        base_gdf = gpd.read_file(first_file)
        # FIXED: Use 'id' instead of 'segment_id'
        base_geometry = base_gdf[['id', 'geometry']].copy()
        print(f"Base geometry extracted from {first_file}")
        print(f"Number of segments: {len(base_geometry)}")
    except KeyError as e:
        print(f"Error: The first file {first_file} does not contain required columns: {str(e)}")
        print(f"Available columns: {list(base_gdf.columns)}")
        return
    except Exception as e:
        print(f"Error reading the first file {first_file}: {str(e)}")
        return

    # Get all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))
    all_data = []

    print(f"Processing {len(gpkg_files)} files...")

    for i, file in enumerate(gpkg_files):
        try:
            gdf = gpd.read_file(file)
            # FIXED: Use 'id' instead of 'segment_id'
            df = gdf[['id', traffic_column]].copy()

            # Extract timestamp from filename
            timestamp = extract_timestamp_from_filename(file)
            if timestamp:
                df['timestamp'] = timestamp
                all_data.append(df)

            # Progress indicator
            if (i + 1) % 10 == 0:
                print(f"Processed {i + 1}/{len(gpkg_files)} files...")

        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

    if not all_data:
        print(f"No valid data found in {city_folder}")
        return

    # Combine all data
    print("Combining all data...")
    combined_data = pd.concat(all_data, ignore_index=True)
    print(f"Combined data shape: {combined_data.shape}")

    # Add hour and time period
    combined_data['hour'] = combined_data['timestamp'].dt.hour
    combined_data['time_period'] = combined_data['hour'].apply(get_time_period)

    # Show time period distribution
    print("\nTime period distribution:")
    print(combined_data['time_period'].value_counts().sort_index())

    # Process each time period
    for period in combined_data['time_period'].unique():
        print(f"\nProcessing time period: {period}")

        # Filter data for this time period
        period_data = combined_data[combined_data['time_period'] == period]
        print(f"Records for {period}: {len(period_data)}")

        # Calculate statistics for each segment
        # FIXED: Use 'id' instead of 'segment_id' in groupby
        period_stats = period_data.groupby('id').agg({
            traffic_column: ['mean', 'std', 'count', 'min', 'max']
        }).round(2)

        # Flatten column names
        period_stats.columns = [f"{traffic_column}_{col}" for col in ['mean', 'std', 'count', 'min', 'max']]
        period_stats = period_stats.reset_index()

        # Merge with geometry
        # FIXED: Use 'id' instead of 'segment_id' in merge
        period_gdf = base_geometry.merge(period_stats, on='id', how='left')

        # Save as GPKG
        output_file = os.path.join(output_folder, f"{period}_{city}.gpkg")
        period_gdf.to_file(output_file, driver='GPKG')
        print(f"Saved {output_file} with {len(period_gdf)} segments")

    print("\nAnalysis complete!")

# Example usage
if __name__ == "__main__":
    city_folder = "/content/traffic-data/traffic_data_smg"  # Change this to your folder path
    traffic_column = "jam_factor"
    output_folder = "/content/traffic-data"  # Change this to your desired output folder

    analyze_city_traffic_by_period(city_folder, traffic_column, output_folder)

Base geometry extracted from /content/traffic-data/traffic_data_smg/semarang_traffic_20250515_190001.gpkg
Number of segments: 1080
Processing 7699 files...
Processed 10/7699 files...
Processed 20/7699 files...
Processed 30/7699 files...
Processed 40/7699 files...
Processed 50/7699 files...
Processed 60/7699 files...
Processed 70/7699 files...
Processed 80/7699 files...
Processed 90/7699 files...
Processed 100/7699 files...
Processed 110/7699 files...
Processed 120/7699 files...
Processed 130/7699 files...
Processed 140/7699 files...
Processed 150/7699 files...
Processed 160/7699 files...
Processed 170/7699 files...
Processed 180/7699 files...
Processed 190/7699 files...
Processed 200/7699 files...
Processed 210/7699 files...
Processed 220/7699 files...
Processed 230/7699 files...
Processed 240/7699 files...
Processed 250/7699 files...
Processed 260/7699 files...
Processed 270/7699 files...
Processed 280/7699 files...
Processed 290/7699 files...
Processed 300/7699 files...
Processed 310

In [14]:
# Semarang aggregation by period - FIXED VERSION

import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime

def extract_timestamp_from_filename(filename):
    """
    Extract timestamp from filename pattern: city_traffic_YYYYMMDD_HHMMSS.gpkg
    """
    try:
        # Get just the filename without path and remove extension
        base_name = os.path.splitext(os.path.basename(filename))[0]
        # Split by underscore and get date and time parts
        parts = base_name.split('_')
        date_str = parts[2]
        time_str = parts[3]
        # Combine date and time and parse
        datetime_str = f"{date_str}_{time_str}"
        return datetime.strptime(datetime_str, "%Y%m%d_%H%M%S")
    except Exception as e:
        print(f"Error parsing timestamp from {filename}: {str(e)}")
        return None

def get_time_period(hour):
    """
    Define time periods in a day
    """
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 9:
        return 'morning_peak'
    elif 9 <= hour < 12:
        return 'morning_offpeak'
    elif 12 <= hour < 14:
        return 'lunch_hours'
    elif 14 <= hour < 16:
        return 'afternoon_offpeak'
    elif 16 <= hour < 19:
        return 'evening_peak'
    elif 19 <= hour < 22:
        return 'evening_offpeak'
    else:
        return 'late_night'

def analyze_city_traffic_by_period(city_folder, traffic_column, output_folder):
    """
    Analyze traffic patterns and save GPKG files for each time period

    Parameters:
    city_folder (str): Path to the folder containing city's GPKG files
    traffic_column (str): Name of the column containing traffic data
    output_folder (str): Path to save the output GPKG files
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get city name from folder name
    city = os.path.basename(city_folder).split('_')[-1]  # Gets 'smg' from 'traffic_data_smg'

    # Get list of all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))

    # Check if any files were found
    if not gpkg_files:
        print(f"No GPKG files found in {city_folder}")
        return

    # Get first file to get the geometry
    try:
        first_file = gpkg_files[0]
        base_gdf = gpd.read_file(first_file)
        # FIXED: Use 'fid' instead of 'id'
        base_geometry = base_gdf[['fid', 'geometry']].copy()
        print(f"Base geometry extracted from {first_file}")
        print(f"Number of segments: {len(base_geometry)}")
    except KeyError as e:
        print(f"Error: The first file {first_file} does not contain required columns: {str(e)}")
        print(f"Available columns: {list(base_gdf.columns)}")
        return
    except Exception as e:
        print(f"Error reading the first file {first_file}: {str(e)}")
        return

    # Get all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))
    all_data = []

    print(f"Processing {len(gpkg_files)} files...")

    for i, file in enumerate(gpkg_files):
        try:
            gdf = gpd.read_file(file)
            # FIXED: Use 'fid' instead of 'id'
            df = gdf[['fid', traffic_column]].copy()

            # Extract timestamp from filename
            timestamp = extract_timestamp_from_filename(file)
            if timestamp:
                df['timestamp'] = timestamp
                all_data.append(df)

            # Progress indicator
            if (i + 1) % 10 == 0:
                print(f"Processed {i + 1}/{len(gpkg_files)} files...")

        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

    if not all_data:
        print(f"No valid data found in {city_folder}")
        return

    # Combine all data
    print("Combining all data...")
    combined_data = pd.concat(all_data, ignore_index=True)
    print(f"Combined data shape: {combined_data.shape}")

    # Add hour and time period
    combined_data['hour'] = combined_data['timestamp'].dt.hour
    combined_data['time_period'] = combined_data['hour'].apply(get_time_period)

    # Show time period distribution
    print("\nTime period distribution:")
    period_counts = combined_data['time_period'].value_counts().sort_index()
    print(period_counts)

    # Show timestamp distribution
    print(f"\nTimestamp range: {combined_data['timestamp'].min()} to {combined_data['timestamp'].max()}")
    print(f"Unique timestamps: {combined_data['timestamp'].nunique()}")

    # Show sample of data
    print(f"\nSample of combined data:")
    print(combined_data.head(10))

    # Process each time period
    for period in combined_data['time_period'].unique():
        print(f"\nProcessing time period: {period}")

        # Filter data for this time period
        period_data = combined_data[combined_data['time_period'] == period]
        print(f"Records for {period}: {len(period_data)}")

        # Calculate statistics for each segment
        # FIXED: Use 'fid' instead of 'id' in groupby
        period_stats = period_data.groupby('fid').agg({
            traffic_column: ['mean', 'std', 'count', 'min', 'max']
        }).round(2)

        # Show some debugging info
        print(f"Sample of period_data for {period}:")
        sample_segment = period_data['fid'].iloc[0]
        sample_data = period_data[period_data['fid'] == sample_segment]
        print(f"Segment {sample_segment} has {len(sample_data)} observations:")
        print(sample_data[['fid', traffic_column, 'timestamp']].head())

        # Check if we have multiple observations per segment
        obs_per_segment = period_data.groupby('fid').size()
        print(f"Observations per segment - Min: {obs_per_segment.min()}, Max: {obs_per_segment.max()}, Mean: {obs_per_segment.mean():.2f}")

        # Flatten column names
        period_stats.columns = [f"{traffic_column}_{col}" for col in ['mean', 'std', 'count', 'min', 'max']]
        period_stats = period_stats.reset_index()

        # Show sample of statistics
        print(f"Sample statistics for {period}:")
        print(period_stats.head())

        # Merge with geometry
        # FIXED: Use 'fid' instead of 'id' in merge
        period_gdf = base_geometry.merge(period_stats, on='fid', how='left')

        # Save as GPKG
        output_file = os.path.join(output_folder, f"{period}_{city}.gpkg")
        period_gdf.to_file(output_file, driver='GPKG')
        print(f"Saved {output_file} with {len(period_gdf)} segments")

    print("\nAnalysis complete!")

# Example usage
if __name__ == "__main__":
    city_folder = "/content/traffic-data/traffic_data_smg"  # Change this to your folder path
    traffic_column = "jam_factor"
    output_folder = "/content/traffic-data"  # Change this to your desired output folder

    analyze_city_traffic_by_period(city_folder, traffic_column, output_folder)

Error: The first file /content/traffic-data/traffic_data_smg/semarang_traffic_20250515_190001.gpkg does not contain required columns: "['fid'] not in index"
Available columns: ['id', 'speed', 'speed_uncapped', 'free_flow', 'jam_factor', 'confidence', 'traversability', 'timestamp', 'geometry']


In [18]:
# Traffic Analysis with Spatially Consistent Segment IDs

import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime

def extract_timestamp_from_filename(filename):
    """
    Extract timestamp from filename pattern: city_traffic_YYYYMMDD_HHMMSS.gpkg
    """
    try:
        # Get just the filename without path and remove extension
        base_name = os.path.splitext(os.path.basename(filename))[0]
        # Split by underscore and get date and time parts
        parts = base_name.split('_')
        date_str = parts[2]
        time_str = parts[3]
        # Combine date and time and parse
        datetime_str = f"{date_str}_{time_str}"
        return datetime.strptime(datetime_str, "%Y%m%d_%H%M%S")
    except Exception as e:
        print(f"Error parsing timestamp from {filename}: {str(e)}")
        return None

def get_time_period(hour):
    """
    Define time periods in a day
    """
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 9:
        return 'morning_peak'
    elif 9 <= hour < 12:
        return 'morning_offpeak'
    elif 12 <= hour < 14:
        return 'lunch_hours'
    elif 14 <= hour < 16:
        return 'afternoon_offpeak'
    elif 16 <= hour < 19:
        return 'evening_peak'
    elif 19 <= hour < 22:
        return 'evening_offpeak'
    else:
        return 'late_night'

def create_spatial_segment_mapping(reference_file):
    """
    Create a spatial segment mapping from a reference file
    This creates a master geometry template with consistent segment IDs
    """
    print(f"Creating spatial segment ID mapping from reference file: {reference_file}")

    try:
        # Read reference file
        ref_gdf = gpd.read_file(reference_file)

        # Create segment_id based on row position
        ref_gdf['segment_id'] = range(1, len(ref_gdf) + 1)

        # Create the master template with geometry and segment_id
        segment_template = ref_gdf[['segment_id', 'geometry']].copy()

        print(f"Created spatial template with {len(segment_template)} segments")
        print(f"Segment IDs range from 1 to {len(segment_template)}")

        return segment_template

    except Exception as e:
        print(f"Error creating spatial segment mapping: {str(e)}")
        return None

def assign_segment_ids_spatially(gdf, segment_template, tolerance=1e-6):
    """
    Assign segment IDs to a GeoDataFrame based on spatial matching with the template
    """
    try:
        # Create a copy to avoid modifying original
        result_gdf = gdf.copy()

        # Method 1: Try spatial join based on geometry overlap/intersection
        # This assumes segments with same geometry get same ID
        joined = gpd.sjoin(result_gdf, segment_template, how='left', predicate='intersects')

        # If spatial join worked well (most segments matched)
        if joined['segment_id'].notna().sum() > len(result_gdf) * 0.8:
            print(f"Spatial join successful: {joined['segment_id'].notna().sum()}/{len(result_gdf)} segments matched")
            result_gdf['segment_id'] = joined['segment_id']
        else:
            # Method 2: Use nearest neighbor matching if spatial join didn't work well
            print("Spatial join didn't work well, using nearest neighbor matching...")

            # Get centroids for faster matching
            gdf_centroids = result_gdf.geometry.centroid
            template_centroids = segment_template.geometry.centroid

            # Find nearest template segment for each segment in current file
            segment_ids = []
            for geom in gdf_centroids:
                # Calculate distances to all template centroids
                distances = template_centroids.distance(geom)
                # Find the closest template segment
                closest_idx = distances.idxmin()
                closest_segment_id = segment_template.loc[closest_idx, 'segment_id']
                segment_ids.append(closest_segment_id)

            result_gdf['segment_id'] = segment_ids
            print(f"Nearest neighbor matching completed for {len(result_gdf)} segments")

        return result_gdf

    except Exception as e:
        print(f"Error in spatial segment ID assignment: {str(e)}")
        # Fallback: assign sequential IDs
        result_gdf = gdf.copy()
        result_gdf['segment_id'] = range(1, len(result_gdf) + 1)
        print(f"Fallback: assigned sequential IDs 1-{len(result_gdf)}")
        return result_gdf

def analyze_city_traffic_by_period(city_folder, traffic_column, output_folder):
    """
    Analyze traffic patterns using spatially consistent segment IDs
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get city name from folder name
    city = os.path.basename(city_folder).split('_')[-1]

    # Get list of all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))

    if not gpkg_files:
        print(f"No GPKG files found in {city_folder}")
        return

    # Step 1: Create spatial segment mapping from the first file
    reference_file = gpkg_files[0]
    segment_template = create_spatial_segment_mapping(reference_file)

    if segment_template is None:
        return

    # Step 2: Process all files and assign spatially consistent segment IDs
    all_data = []
    print(f"\nProcessing {len(gpkg_files)} files with spatial matching...")

    for i, file in enumerate(gpkg_files):
        try:
            gdf = gpd.read_file(file)

            # Assign segment_id based on spatial matching
            gdf_with_ids = assign_segment_ids_spatially(gdf, segment_template)

            # Verify we have segment IDs
            if 'segment_id' not in gdf_with_ids.columns:
                print(f"⚠️  Warning: Could not assign segment IDs to {file}")
                continue

            # Extract the data we need
            df = gdf_with_ids[['segment_id', traffic_column]].copy()

            # Extract timestamp from filename
            timestamp = extract_timestamp_from_filename(file)
            if timestamp:
                df['timestamp'] = timestamp
                all_data.append(df)

            # Progress indicator
            if (i + 1) % 100 == 0:
                print(f"Processed {i + 1}/{len(gpkg_files)} files...")

        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

    if not all_data:
        print(f"No valid data found in {city_folder}")
        return

    # Step 3: Combine all data
    print("\nCombining all data...")
    combined_data = pd.concat(all_data, ignore_index=True)
    print(f"Combined data shape: {combined_data.shape}")

    # Add hour and time period
    combined_data['hour'] = combined_data['timestamp'].dt.hour
    combined_data['time_period'] = combined_data['hour'].apply(get_time_period)

    # Show time period distribution
    print("\nTime period distribution:")
    period_counts = combined_data['time_period'].value_counts().sort_index()
    print(period_counts)

    # Show timestamp distribution
    print(f"\nTimestamp range: {combined_data['timestamp'].min()} to {combined_data['timestamp'].max()}")
    print(f"Unique timestamps: {combined_data['timestamp'].nunique()}")

    # Show segment distribution
    print(f"Unique segments: {combined_data['segment_id'].nunique()}")
    print(f"Segment ID range: {combined_data['segment_id'].min()} to {combined_data['segment_id'].max()}")

    # Step 4: Process each time period
    for period in combined_data['time_period'].unique():
        print(f"\n{'='*50}")
        print(f"Processing time period: {period}")
        print(f"{'='*50}")

        # Filter data for this time period
        period_data = combined_data[combined_data['time_period'] == period]
        print(f"Records for {period}: {len(period_data)}")

        # Calculate statistics for each segment
        period_stats = period_data.groupby('segment_id').agg({
            traffic_column: ['mean', 'std', 'count', 'min', 'max']
        }).round(2)

        # Show some debugging info
        print(f"\nSample segment analysis for {period}:")
        sample_segments = [1, 2, 3, 4, 5]  # Check first 5 segments

        for seg_id in sample_segments:
            if seg_id in period_data['segment_id'].values:
                sample_data = period_data[period_data['segment_id'] == seg_id]
                values = sample_data[traffic_column].tolist()
                print(f"Segment {seg_id}: {len(values)} observations")
                print(f"  Values: {values[:5]}{'...' if len(values) > 5 else ''}")  # Show first 5 values
                print(f"  Mean: {sample_data[traffic_column].mean():.2f}")
                print(f"  Std: {sample_data[traffic_column].std():.2f}")

        # Check observations per segment
        obs_per_segment = period_data.groupby('segment_id').size()
        print(f"\nObservations per segment - Min: {obs_per_segment.min()}, Max: {obs_per_segment.max()}, Mean: {obs_per_segment.mean():.2f}")

        # Flatten column names
        period_stats.columns = [f"{traffic_column}_{col}" for col in ['mean', 'std', 'count', 'min', 'max']]
        period_stats = period_stats.reset_index()

        # Show sample statistics to verify variation
        print(f"\nFirst 5 segment statistics:")
        print(period_stats.head()[['segment_id', f'{traffic_column}_mean', f'{traffic_column}_std', f'{traffic_column}_count']])

        # Show variation in means to verify segments have different values
        means = period_stats[f'{traffic_column}_mean']
        print(f"\nVariation in segment means:")
        print(f"  Min mean: {means.min():.2f}")
        print(f"  Max mean: {means.max():.2f}")
        print(f"  Mean of means: {means.mean():.2f}")
        print(f"  Std of means: {means.std():.2f}")

        # Merge with geometry from template
        period_gdf = segment_template.merge(period_stats, on='segment_id', how='left')

        # Save as GPKG
        output_file = os.path.join(output_folder, f"{period}_{city}.gpkg")
        period_gdf.to_file(output_file, driver='GPKG')
        print(f"Saved {output_file} with {len(period_gdf)} segments")

    print("\n" + "="*50)
    print("Analysis complete!")
    print("="*50)

# Example usage
if __name__ == "__main__":
    city_folder = "/content/traffic-data/traffic_data_smg"
    traffic_column = "jam_factor"
    output_folder = "/content/traffic-data"

    analyze_city_traffic_by_period(city_folder, traffic_column, output_folder)

Streaming output truncated to the last 5000 lines.
Fallback: assigned sequential IDs 1-1081
Spatial join successful: 6952/1077 segments matched
Error in spatial segment ID assignment: cannot reindex on an axis with duplicate labels
Fallback: assigned sequential IDs 1-1077
Spatial join successful: 6952/1077 segments matched
Error in spatial segment ID assignment: cannot reindex on an axis with duplicate labels
Fallback: assigned sequential IDs 1-1077
Spatial join successful: 6950/1076 segments matched
Error in spatial segment ID assignment: cannot reindex on an axis with duplicate labels
Fallback: assigned sequential IDs 1-1076
Spatial join successful: 6945/1075 segments matched
Error in spatial segment ID assignment: cannot reindex on an axis with duplicate labels
Fallback: assigned sequential IDs 1-1075
Spatial join successful: 6962/1079 segments matched
Error in spatial segment ID assignment: cannot reindex on an axis with duplicate labels
Fallback: assigned sequential IDs 1-1079
Spa

In [1]:
# Using GDAL to aggregate
# Traffic Analysis using GDAL/OGR to read GPKG files

import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime
from osgeo import ogr, osr
import geopandas as gpd
from shapely.wkt import loads

def extract_timestamp_from_filename(filename):
    """
    Extract timestamp from filename pattern: city_traffic_YYYYMMDD_HHMMSS.gpkg
    """
    try:
        # Get just the filename without path and remove extension
        base_name = os.path.splitext(os.path.basename(filename))[0]
        # Split by underscore and get date and time parts
        parts = base_name.split('_')
        date_str = parts[2]
        time_str = parts[3]
        # Combine date and time and parse
        datetime_str = f"{date_str}_{time_str}"
        return datetime.strptime(datetime_str, "%Y%m%d_%H%M%S")
    except Exception as e:
        print(f"Error parsing timestamp from {filename}: {str(e)}")
        return None

def get_time_period(hour):
    """
    Define time periods in a day
    """
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 9:
        return 'morning_peak'
    elif 9 <= hour < 12:
        return 'morning_offpeak'
    elif 12 <= hour < 14:
        return 'lunch_hours'
    elif 14 <= hour < 16:
        return 'afternoon_offpeak'
    elif 16 <= hour < 19:
        return 'evening_peak'
    elif 19 <= hour < 22:
        return 'evening_offpeak'
    else:
        return 'late_night'

def read_gpkg_with_gdal(filepath, traffic_column):
    """
    Read GPKG file using GDAL/OGR and extract FID, traffic data, and geometry
    """
    try:
        # Open the GPKG file
        driver = ogr.GetDriverByName("GPKG")
        datasource = driver.Open(filepath, 0)  # 0 means read-only

        if datasource is None:
            print(f"Could not open {filepath}")
            return None

        # Get the layer (assuming first layer)
        layer = datasource.GetLayer(0)
        layer_name = layer.GetName()
        feature_count = layer.GetFeatureCount()

        print(f"Reading {filepath}")
        print(f"  Layer: {layer_name}")
        print(f"  Features: {feature_count}")

        # Get layer definition to see available fields
        layer_defn = layer.GetLayerDefn()
        field_names = []
        for i in range(layer_defn.GetFieldCount()):
            field_defn = layer_defn.GetFieldDefn(i)
            field_names.append(field_defn.GetName())

        print(f"  Available fields: {field_names}")

        # Check if traffic column exists
        if traffic_column not in field_names:
            print(f"  ❌ Traffic column '{traffic_column}' not found")
            return None

        # Read all features
        data_rows = []
        layer.ResetReading()  # Start from beginning

        for feature in layer:
            # Get FID (this is the unique feature identifier)
            fid = feature.GetFID()

            # Get traffic value
            traffic_value = feature.GetField(traffic_column)

            # Get geometry as WKT
            geometry = feature.GetGeometryRef()
            if geometry:
                geom_wkt = geometry.ExportToWkt()
            else:
                geom_wkt = None

            data_rows.append({
                'fid': fid,
                traffic_column: traffic_value,
                'geometry_wkt': geom_wkt
            })

        # Clean up
        datasource = None

        if data_rows:
            df = pd.DataFrame(data_rows)
            print(f"  ✅ Successfully read {len(df)} features")
            print(f"  FID range: {df['fid'].min()} to {df['fid'].max()}")
            print(f"  FID unique count: {df['fid'].nunique()}")
            print(f"  Sample FIDs: {df['fid'].head().tolist()}")
            return df
        else:
            print(f"  ❌ No features read")
            return None

    except Exception as e:
        print(f"Error reading {filepath} with GDAL: {str(e)}")
        return None

def create_reference_geodataframe(reference_df):
    """
    Convert the reference DataFrame with WKT geometries to a GeoDataFrame
    """
    try:
        from shapely.wkt import loads

        # Convert WKT to Shapely geometries
        geometries = [loads(wkt) if wkt else None for wkt in reference_df['geometry_wkt']]

        # Create GeoDataFrame
        gdf = gpd.GeoDataFrame(
            reference_df[['fid']],
            geometry=geometries
        )

        # Remove any features with null geometry
        gdf = gdf[gdf.geometry.notna()]

        print(f"Created reference GeoDataFrame with {len(gdf)} features")
        return gdf

    except Exception as e:
        print(f"Error creating reference GeoDataFrame: {str(e)}")
        return None

def analyze_city_traffic_by_period_gdal(city_folder, traffic_column, output_folder):
    """
    Analyze traffic patterns using GDAL/OGR to read GPKG files
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get city name from folder name
    city = os.path.basename(city_folder).split('_')[-1]

    # Get list of all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))

    if not gpkg_files:
        print(f"No GPKG files found in {city_folder}")
        return

    print(f"Found {len(gpkg_files)} GPKG files")

    # Step 1: Read reference file to get geometry template
    reference_file = gpkg_files[0]
    print(f"\n{'='*60}")
    print(f"CREATING REFERENCE FROM: {os.path.basename(reference_file)}")
    print(f"{'='*60}")

    reference_df = read_gpkg_with_gdal(reference_file, traffic_column)
    if reference_df is None:
        return

    # Create reference GeoDataFrame for output
    reference_gdf = create_reference_geodataframe(reference_df)
    if reference_gdf is None:
        return

    # Step 2: Read all files using GDAL
    all_data = []
    print(f"\n{'='*60}")
    print(f"PROCESSING ALL FILES")
    print(f"{'='*60}")

    for i, file in enumerate(gpkg_files):
        df = read_gpkg_with_gdal(file, traffic_column)

        if df is not None:
            # Extract timestamp from filename
            timestamp = extract_timestamp_from_filename(file)
            if timestamp:
                df['timestamp'] = timestamp
                # Keep only the columns we need
                df_clean = df[['fid', traffic_column, 'timestamp']].copy()
                all_data.append(df_clean)

        # Progress indicator
        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(gpkg_files)} files...")

    if not all_data:
        print(f"No valid data found in {city_folder}")
        return

    # Step 3: Combine all data
    print(f"\n{'='*60}")
    print(f"COMBINING AND ANALYZING DATA")
    print(f"{'='*60}")

    combined_data = pd.concat(all_data, ignore_index=True)
    print(f"Combined data shape: {combined_data.shape}")

    # Add hour and time period
    combined_data['hour'] = combined_data['timestamp'].dt.hour
    combined_data['time_period'] = combined_data['hour'].apply(get_time_period)

    # Show detailed data summary
    print(f"\nData Summary:")
    print(f"  Total records: {len(combined_data)}")
    print(f"  Unique FIDs: {combined_data['fid'].nunique()}")
    print(f"  FID range: {combined_data['fid'].min()} to {combined_data['fid'].max()}")
    print(f"  Unique timestamps: {combined_data['timestamp'].nunique()}")
    print(f"  Timestamp range: {combined_data['timestamp'].min()} to {combined_data['timestamp'].max()}")

    # Show time period distribution
    print(f"\nTime period distribution:")
    period_counts = combined_data['time_period'].value_counts().sort_index()
    print(period_counts)

    # Step 4: Process each time period
    for period in combined_data['time_period'].unique():
        print(f"\n{'='*60}")
        print(f"PROCESSING TIME PERIOD: {period}")
        print(f"{'='*60}")

        # Filter data for this time period
        period_data = combined_data[combined_data['time_period'] == period]
        print(f"Records for {period}: {len(period_data)}")

        # Calculate statistics for each segment using FID
        period_stats = period_data.groupby('fid').agg({
            traffic_column: ['mean', 'std', 'count', 'min', 'max']
        }).round(2)

        # Show detailed debugging info
        print(f"\nDetailed segment analysis for {period}:")
        sample_fids = [1, 2, 3, 4, 5]  # Check first 5 FIDs

        for fid in sample_fids:
            if fid in period_data['fid'].values:
                sample_data = period_data[period_data['fid'] == fid]
                values = sample_data[traffic_column].tolist()
                print(f"FID {fid}: {len(values)} observations")
                print(f"  Values: {values[:10]}{'...' if len(values) > 10 else ''}")  # Show first 10 values
                print(f"  Mean: {sample_data[traffic_column].mean():.3f}")
                print(f"  Std: {sample_data[traffic_column].std():.3f}")

        # Check observations per segment
        obs_per_segment = period_data.groupby('fid').size()
        print(f"\nObservations per FID - Min: {obs_per_segment.min()}, Max: {obs_per_segment.max()}, Mean: {obs_per_segment.mean():.2f}")

        # Flatten column names
        period_stats.columns = [f"{traffic_column}_{col}" for col in ['mean', 'std', 'count', 'min', 'max']]
        period_stats = period_stats.reset_index()

        # Show sample statistics to verify variation
        print(f"\nFirst 10 FID statistics:")
        display_cols = ['fid', f'{traffic_column}_mean', f'{traffic_column}_std', f'{traffic_column}_count']
        print(period_stats.head(10)[display_cols])

        # Show variation in means to verify FIDs have different values
        means = period_stats[f'{traffic_column}_mean']
        print(f"\nVariation in FID means:")
        print(f"  Min mean: {means.min():.3f}")
        print(f"  Max mean: {means.max():.3f}")
        print(f"  Mean of means: {means.mean():.3f}")
        print(f"  Std of means: {means.std():.3f}")
        print(f"  Unique mean values: {means.nunique()}")

        # Merge with geometry from reference
        period_gdf = reference_gdf.merge(period_stats, on='fid', how='left')

        # Count how many FIDs have data
        valid_data_count = period_gdf[f'{traffic_column}_mean'].notna().sum()
        print(f"\nGeometry merge result:")
        print(f"  Total geometries: {len(period_gdf)}")
        print(f"  FIDs with data: {valid_data_count}")
        print(f"  FIDs without data: {len(period_gdf) - valid_data_count}")

        # Save as GPKG
        output_file = os.path.join(output_folder, f"{period}_{city}_gdal.gpkg")
        period_gdf.to_file(output_file, driver='GPKG')
        print(f"Saved {output_file}")

    print(f"\n{'='*60}")
    print("GDAL ANALYSIS COMPLETE!")
    print(f"{'='*60}")

# Example usage
if __name__ == "__main__":
    city_folder = "/content/traffic-data/traffic_data_smg"
    traffic_column = "jam_factor"
    output_folder = "/content/traffic-data"

    analyze_city_traffic_by_period_gdal(city_folder, traffic_column, output_folder)

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime

def extract_timestamp_from_filename(filename):
    """
    Extract timestamp from filename pattern: city_traffic_YYYYMMDD_HHMMSS.gpkg
    """
    try:
        # Get just the filename without path and remove extension
        base_name = os.path.splitext(os.path.basename(filename))[0]
        # Split by underscore and get date and time parts
        parts = base_name.split('_')
        date_str = parts[2]
        time_str = parts[3]
        # Combine date and time and parse
        datetime_str = f"{date_str}_{time_str}"
        return datetime.strptime(datetime_str, "%Y%m%d_%H%M%S")
    except Exception as e:
        print(f"Error parsing timestamp from {filename}: {str(e)}")
        return None

def get_time_period(hour):
    """
    Define time periods in a day
    """
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 9:
        return 'morning_peak'
    elif 9 <= hour < 12:
        return 'morning_offpeak'
    elif 12 <= hour < 14:
        return 'lunch_hours'
    elif 14 <= hour < 16:
        return 'afternoon_offpeak'
    elif 16 <= hour < 19:
        return 'evening_peak'
    elif 19 <= hour < 22:
        return 'evening_offpeak'
    else:
        return 'late_night'

def analyze_city_traffic_by_period(city_folder, traffic_column, output_folder):
    """
    Analyze traffic patterns and save GPKG files for each time period

    Parameters:
    city_folder (str): Path to the folder containing city's GPKG files
    traffic_column (str): Name of the column containing traffic data
    output_folder (str): Path to save the output GPKG files
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get city name from folder name
    city = os.path.basename(city_folder).split('_')[-1]  # Gets 'smg' from 'traffic_data_smg'

    # Get first file to get the geometry
    first_file = glob.glob(os.path.join(city_folder, "*.gpkg"))[0]
    base_gdf = gpd.read_file(first_file)
    base_geometry = base_gdf[['segment_id', 'geometry']].copy()

    # Get all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))
    all_data = []

    for file in gpkg_files:
        try:
            gdf = gpd.read_file(file)
            df = gdf[['segment_id', traffic_column]].copy()

            # Extract timestamp from filename
            timestamp = extract_timestamp_from_filename(file)
            if timestamp:
                df['timestamp'] = timestamp
                all_data.append(df)

        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

    if not all_data:
        print(f"No valid data found in {city_folder}")
        return

    # Combine all data
    combined_data = pd.concat(all_data, ignore_index=True)

    # Add hour and time period
    combined_data['hour'] = combined_data['timestamp'].dt.hour
    combined_data['time_period'] = combined_data['hour'].apply(get_time_period)

    # Process each time period
    for period in combined_data['time_period'].unique():
        # Filter data for this time period
        period_data = combined_data[combined_data['time_period'] == period]

        # Calculate statistics for each segment
        period_stats = period_data.groupby('segment_id').agg({
            traffic_column: ['mean', 'std', 'count', 'min', 'max']
        }).round(2)

        # Flatten column names
        period_stats.columns = [f"{traffic_column}_{col}" for col in ['mean', 'std', 'count', 'min', 'max']]
        period_stats = period_stats.reset_index()

        # Merge with geometry
        period_gdf = base_geometry.merge(period_stats, on='segment_id', how='left')

        # Save as GPKG
        output_file = os.path.join(output_folder, f"{period}_{city}.gpkg")
        period_gdf.to_file(output_file, driver='GPKG')
        print(f"Saved {output_file}")

# Example usage
if __name__ == "__main__":
    city_folder = "traffic_data_bdg"  # Change this to your folder path
    traffic_column = "jam_factor"
    output_folder = "traffic_bdg_output"  # Change this to your desired output folder

    analyze_city_traffic_by_period(city_folder, traffic_column, output_folder)

Saved traffic_bdg_output/morning_peak_bdg.gpkg
Saved traffic_bdg_output/evening_peak_bdg.gpkg
Saved traffic_bdg_output/evening_offpeak_bdg.gpkg
Saved traffic_bdg_output/night_bdg.gpkg
Saved traffic_bdg_output/late_night_bdg.gpkg
Saved traffic_bdg_output/morning_offpeak_bdg.gpkg
Saved traffic_bdg_output/afternoon_offpeak_bdg.gpkg
Saved traffic_bdg_output/lunch_hours_bdg.gpkg


In [ ]:
# Jakarta aggregation by period

import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime

def extract_timestamp_from_filename(filename):
    """
    Extract timestamp from filename pattern: city_traffic_YYYYMMDD_HHMMSS.gpkg
    """
    try:
        # Get just the filename without path and remove extension
        base_name = os.path.splitext(os.path.basename(filename))[0]
        # Split by underscore and get date and time parts
        parts = base_name.split('_')
        date_str = parts[2]
        time_str = parts[3]
        # Combine date and time and parse
        datetime_str = f"{date_str}_{time_str}"
        return datetime.strptime(datetime_str, "%Y%m%d_%H%M%S")
    except Exception as e:
        print(f"Error parsing timestamp from {filename}: {str(e)}")
        return None

def get_time_period(hour):
    """
    Define time periods in a day
    """
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 9:
        return 'morning_peak'
    elif 9 <= hour < 12:
        return 'morning_offpeak'
    elif 12 <= hour < 14:
        return 'lunch_hours'
    elif 14 <= hour < 16:
        return 'afternoon_offpeak'
    elif 16 <= hour < 19:
        return 'evening_peak'
    elif 19 <= hour < 22:
        return 'evening_offpeak'
    else:
        return 'late_night'

def analyze_city_traffic_by_period(city_folder, traffic_column, output_folder):
    """
    Analyze traffic patterns and save GPKG files for each time period

    Parameters:
    city_folder (str): Path to the folder containing city's GPKG files
    traffic_column (str): Name of the column containing traffic data
    output_folder (str): Path to save the output GPKG files
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get city name from folder name
    city = os.path.basename(city_folder).split('_')[-1]  # Gets 'smg' from 'traffic_data_smg'

    # Get first file to get the geometry
    first_file = glob.glob(os.path.join(city_folder, "*.gpkg"))[0]
    base_gdf = gpd.read_file(first_file)
    base_geometry = base_gdf[['segment_id', 'geometry']].copy()

    # Get all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))
    all_data = []

    for file in gpkg_files:
        try:
            gdf = gpd.read_file(file)
            df = gdf[['segment_id', traffic_column]].copy()

            # Extract timestamp from filename
            timestamp = extract_timestamp_from_filename(file)
            if timestamp:
                df['timestamp'] = timestamp
                all_data.append(df)

        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

    if not all_data:
        print(f"No valid data found in {city_folder}")
        return

    # Combine all data
    combined_data = pd.concat(all_data, ignore_index=True)

    # Add hour and time period
    combined_data['hour'] = combined_data['timestamp'].dt.hour
    combined_data['time_period'] = combined_data['hour'].apply(get_time_period)

    # Process each time period
    for period in combined_data['time_period'].unique():
        # Filter data for this time period
        period_data = combined_data[combined_data['time_period'] == period]

        # Calculate statistics for each segment
        period_stats = period_data.groupby('segment_id').agg({
            traffic_column: ['mean', 'std', 'count', 'min', 'max']
        }).round(2)

        # Flatten column names
        period_stats.columns = [f"{traffic_column}_{col}" for col in ['mean', 'std', 'count', 'min', 'max']]
        period_stats = period_stats.reset_index()

        # Merge with geometry
        period_gdf = base_geometry.merge(period_stats, on='segment_id', how='left')

        # Save as GPKG
        output_file = os.path.join(output_folder, f"{period}_{city}.gpkg")
        period_gdf.to_file(output_file, driver='GPKG')
        print(f"Saved {output_file}")

# Example usage
if __name__ == "__main__":
    city_folder = "traffic_data_jkt"  # Change this to your folder path
    traffic_column = "jam_factor"
    output_folder = "traffic_jkt_output"  # Change this to your desired output folder

    analyze_city_traffic_by_period(city_folder, traffic_column, output_folder)

Error reading traffic_data_jkt/jakarta_traffic_20241209_011838.gpkg: "['segment_id'] not in index"
Saved traffic_jkt_output/morning_peak_jkt.gpkg
Saved traffic_jkt_output/evening_peak_jkt.gpkg
Saved traffic_jkt_output/night_jkt.gpkg
Saved traffic_jkt_output/late_night_jkt.gpkg
Saved traffic_jkt_output/morning_offpeak_jkt.gpkg
Saved traffic_jkt_output/lunch_hours_jkt.gpkg
Saved traffic_jkt_output/evening_offpeak_jkt.gpkg
Saved traffic_jkt_output/afternoon_offpeak_jkt.gpkg


# PERIOD BASED


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime

In [ ]:
def extract_timestamp_from_filename(filename):
    """
    Extract timestamp from filename pattern: city_traffic_YYYYMMDD_HHMMSS
    """
    # Get the date and time parts from filename
    try:
        parts = os.path.basename(filename).split('_')
        date_str = parts[2]
        time_str = parts[3]
        # Combine date and time and parse
        datetime_str = f"{date_str}_{time_str}"
        return datetime.strptime(datetime_str, "%Y%m%d_%H%M%S")
    except Exception as e:
        print(f"Error parsing timestamp from {filename}: {str(e)}")
        return None

def get_time_period(hour):
    """
    Define time periods in a day
    """
    if 0 <= hour < 6:
        return 'Night (00:00-06:00)'
    elif 6 <= hour < 9:
        return 'Morning Peak (06:00-09:00)'
    elif 9 <= hour < 12:
        return 'Morning Off-peak (09:00-12:00)'
    elif 12 <= hour < 14:
        return 'Lunch Hours (12:00-14:00)'
    elif 14 <= hour < 16:
        return 'Afternoon Off-peak (14:00-16:00)'
    elif 16 <= hour < 19:
        return 'Evening Peak (16:00-19:00)'
    elif 19 <= hour < 22:
        return 'Evening Off-peak (19:00-22:00)'
    else:
        return 'Late Night (22:00-00:00)'

def analyze_traffic_patterns(folder_path, traffic_column):
    """
    Analyze traffic patterns by time period for different cities
    """
    gpkg_files = glob.glob(os.path.join(folder_path, "*.gpkg"))
    all_data = []

    for file in gpkg_files:
        try:
            gdf = gpd.read_file(file)

            # Extract city name from filename
            city = os.path.basename(file).split('_')[0]

            df = gdf[['segment_id', traffic_column]].copy()
            df['city'] = city

            # Extract timestamp from filename
            timestamp = extract_timestamp_from_filename(file)
            if timestamp:
                df['timestamp'] = timestamp
                all_data.append(df)

        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

    # Combine all data
    combined_data = pd.concat(all_data, ignore_index=True)

    # Extract hour and add time period
    combined_data['hour'] = combined_data['timestamp'].dt.hour
    combined_data['time_period'] = combined_data['hour'].apply(get_time_period)

    # Analyze by city and time period
    city_time_stats = combined_data.groupby(['city', 'time_period']).agg({
        traffic_column: ['mean', 'std', 'count'],
        'segment_id': 'nunique'
    }).round(2)

    # Analyze peak hours for each city
    peak_hours = combined_data.groupby(['city', 'hour'])[traffic_column].mean()
    worst_hours = peak_hours.groupby(level=0).idxmax()
    best_hours = peak_hours.groupby(level=0).idxmin()

    return combined_data, city_time_stats, worst_hours, best_hours

def plot_daily_patterns(combined_data, traffic_column):
    """
    Create hourly traffic pattern plots for each city
    """
    import matplotlib.pyplot as plt

    plt.figure(figsize=(15, 6))

    for city in combined_data['city'].unique():
        city_data = combined_data[combined_data['city'] == city]
        hourly_avg = city_data.groupby('hour')[traffic_column].mean()
        plt.plot(hourly_avg.index, hourly_avg.values, label=city, marker='o')

    plt.xlabel('Hour of Day')
    plt.ylabel('Average Traffic Condition')
    plt.title('Daily Traffic Patterns by City')
    plt.legend()
    plt.grid(True)
    plt.savefig('daily_traffic_patterns.png')
    plt.close()



In [ ]:
# Example usage
if __name__ == "__main__":
    folder_path = "/content/traffic-data/traffic_data_smg"
    traffic_column = "traffic_condition"

    # Analyze patterns
    combined_data, city_time_stats, worst_hours, best_hours = analyze_traffic_patterns(
        folder_path, traffic_column
    )

    # Print results
    print("\nTraffic Analysis by City and Time Period:")
    print(city_time_stats)

    print("\nPeak Traffic Hours by City:")
    for city, (_, hour) in worst_hours.items():
        print(f"{city}: {hour:02d}:00")

    print("\nBest Traffic Hours by City:")
    for city, (_, hour) in best_hours.items():
        print(f"{city}: {hour:02d}:00")

    # Create visualization
    plot_daily_patterns(combined_data, traffic_column)

In [ ]:
# Semarang aggregation by period - FIXED VERSION

import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime

def extract_timestamp_from_filename(filename):
    """
    Extract timestamp from filename pattern: city_traffic_YYYYMMDD_HHMMSS.gpkg
    """
    try:
        # Get just the filename without path and remove extension
        base_name = os.path.splitext(os.path.basename(filename))[0]
        # Split by underscore and get date and time parts
        parts = base_name.split('_')
        date_str = parts[2]
        time_str = parts[3]
        # Combine date and time and parse
        datetime_str = f"{date_str}_{time_str}"
        return datetime.strptime(datetime_str, "%Y%m%d_%H%M%S")
    except Exception as e:
        print(f"Error parsing timestamp from {filename}: {str(e)}")
        return None

def get_time_period(hour):
    """
    Define time periods in a day
    """
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 9:
        return 'morning_peak'
    elif 9 <= hour < 12:
        return 'morning_offpeak'
    elif 12 <= hour < 14:
        return 'lunch_hours'
    elif 14 <= hour < 16:
        return 'afternoon_offpeak'
    elif 16 <= hour < 19:
        return 'evening_peak'
    elif 19 <= hour < 22:
        return 'evening_offpeak'
    else:
        return 'late_night'

def analyze_city_traffic_by_period(city_folder, traffic_column, output_folder):
    """
    Analyze traffic patterns and save GPKG files for each time period

    Parameters:
    city_folder (str): Path to the folder containing city's GPKG files
    traffic_column (str): Name of the column containing traffic data
    output_folder (str): Path to save the output GPKG files
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get city name from folder name
    city = os.path.basename(city_folder).split('_')[-1]  # Gets 'smg' from 'traffic_data_smg'

    # Get list of all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))

    # Check if any files were found
    if not gpkg_files:
        print(f"No GPKG files found in {city_folder}")
        return

    # Get first file to get the geometry
    try:
        first_file = gpkg_files[0]
        base_gdf = gpd.read_file(first_file)
        # FIXED: Use 'id' instead of 'segment_id'
        base_geometry = base_gdf[['id', 'geometry']].copy()
        print(f"Base geometry extracted from {first_file}")
        print(f"Number of segments: {len(base_geometry)}")
    except KeyError as e:
        print(f"Error: The first file {first_file} does not contain required columns: {str(e)}")
        print(f"Available columns: {list(base_gdf.columns)}")
        return
    except Exception as e:
        print(f"Error reading the first file {first_file}: {str(e)}")
        return

    # Get all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))
    all_data = []

    print(f"Processing {len(gpkg_files)} files...")

    for i, file in enumerate(gpkg_files):
        try:
            gdf = gpd.read_file(file)
            # Extract only the columns we need (no timestamp column in data)
            df = gdf[['id', traffic_column]].copy()

            # Extract timestamp from filename
            timestamp = extract_timestamp_from_filename(file)
            if timestamp:
                df['timestamp'] = timestamp
                all_data.append(df)

            # Progress indicator
            if (i + 1) % 10 == 0:
                print(f"Processed {i + 1}/{len(gpkg_files)} files...")

        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

    if not all_data:
        print(f"No valid data found in {city_folder}")
        return

    # Combine all data
    print("Combining all data...")
    combined_data = pd.concat(all_data, ignore_index=True)
    print(f"Combined data shape: {combined_data.shape}")

    # Add hour and time period
    combined_data['hour'] = combined_data['timestamp'].dt.hour
    combined_data['time_period'] = combined_data['hour'].apply(get_time_period)

    # Show time period distribution
    print("\nTime period distribution:")
    period_counts = combined_data['time_period'].value_counts().sort_index()
    print(period_counts)

    # Show timestamp distribution
    print(f"\nTimestamp range: {combined_data['timestamp'].min()} to {combined_data['timestamp'].max()}")
    print(f"Unique timestamps: {combined_data['timestamp'].nunique()}")

    # Show sample of data
    print(f"\nSample of combined data:")
    print(combined_data.head(10))

    # Process each time period
    for period in combined_data['time_period'].unique():
        print(f"\nProcessing time period: {period}")

        # Filter data for this time period
        period_data = combined_data[combined_data['time_period'] == period]
        print(f"Records for {period}: {len(period_data)}")

        # Calculate statistics for each segment
        # FIXED: Use 'id' instead of 'segment_id' in groupby
        period_stats = period_data.groupby('id').agg({
            traffic_column: ['mean', 'std', 'count', 'min', 'max']
        }).round(2)

        # Show some debugging info
        print(f"Sample of period_data for {period}:")
        sample_segment = period_data['id'].iloc[0]
        sample_data = period_data[period_data['id'] == sample_segment]
        print(f"Segment {sample_segment} has {len(sample_data)} observations:")
        print(sample_data[['id', traffic_column, 'timestamp']].head())

        # Check if we have multiple observations per segment
        obs_per_segment = period_data.groupby('id').size()
        print(f"Observations per segment - Min: {obs_per_segment.min()}, Max: {obs_per_segment.max()}, Mean: {obs_per_segment.mean():.2f}")

        # Flatten column names
        period_stats.columns = [f"{traffic_column}_{col}" for col in ['mean', 'std', 'count', 'min', 'max']]
        period_stats = period_stats.reset_index()

        # Show sample of statistics
        print(f"Sample statistics for {period}:")
        print(period_stats.head())

        # Merge with geometry
        # FIXED: Use 'id' instead of 'segment_id' in merge
        period_gdf = base_geometry.merge(period_stats, on='id', how='left')

        # Save as GPKG
        output_file = os.path.join(output_folder, f"{period}_{city}.gpkg")
        period_gdf.to_file(output_file, driver='GPKG')
        print(f"Saved {output_file} with {len(period_gdf)} segments")

    print("\nAnalysis complete!")

# Example usage
if __name__ == "__main__":
    city_folder = "/content/traffic-data/traffic_data_smg"  # Change this to your folder path
    traffic_column = "jam_factor"
    output_folder = "/content/traffic-data"  # Change this to your desired output folder

    analyze_city_traffic_by_period(city_folder, traffic_column, output_folder)

In [3]:
# Traffic Analysis using GDAL/OGR to read GPKG files

import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime
from osgeo import ogr, osr
import geopandas as gpd
from shapely.wkt import loads

def extract_timestamp_from_filename(filename):
    """
    Extract timestamp from filename pattern: city_traffic_YYYYMMDD_HHMMSS.gpkg
    """
    try:
        # Get just the filename without path and remove extension
        base_name = os.path.splitext(os.path.basename(filename))[0]
        # Split by underscore and get date and time parts
        parts = base_name.split('_')
        date_str = parts[2]
        time_str = parts[3]
        # Combine date and time and parse
        datetime_str = f"{date_str}_{time_str}"
        return datetime.strptime(datetime_str, "%Y%m%d_%H%M%S")
    except Exception as e:
        print(f"Error parsing timestamp from {filename}: {str(e)}")
        return None

def get_time_period(hour):
    """
    Define time periods in a day
    """
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 9:
        return 'morning_peak'
    elif 9 <= hour < 12:
        return 'morning_offpeak'
    elif 12 <= hour < 14:
        return 'lunch_hours'
    elif 14 <= hour < 16:
        return 'afternoon_offpeak'
    elif 16 <= hour < 19:
        return 'evening_peak'
    elif 19 <= hour < 22:
        return 'evening_offpeak'
    else:
        return 'late_night'

def read_gpkg_with_gdal(filepath, traffic_column):
    """
    Read GPKG file using GDAL/OGR and extract FID, traffic data, and geometry
    """
    try:
        # Open the GPKG file
        driver = ogr.GetDriverByName("GPKG")
        datasource = driver.Open(filepath, 0)  # 0 means read-only

        if datasource is None:
            print(f"Could not open {filepath}")
            return None

        # Get the layer (assuming first layer)
        layer = datasource.GetLayer(0)
        layer_name = layer.GetName()
        feature_count = layer.GetFeatureCount()

        print(f"Reading {filepath}")
        print(f"  Layer: {layer_name}")
        print(f"  Features: {feature_count}")

        # Get layer definition to see available fields
        layer_defn = layer.GetLayerDefn()
        field_names = []
        for i in range(layer_defn.GetFieldCount()):
            field_defn = layer_defn.GetFieldDefn(i)
            field_names.append(field_defn.GetName())

        print(f"  Available fields: {field_names}")

        # Check if traffic column exists
        if traffic_column not in field_names:
            print(f"  ❌ Traffic column '{traffic_column}' not found")
            return None

        # Read all features
        data_rows = []
        layer.ResetReading()  # Start from beginning

        for feature in layer:
            # Get FID (this is the unique feature identifier)
            fid = feature.GetFID()

            # Get traffic value
            traffic_value = feature.GetField(traffic_column)

            # Get geometry as WKT
            geometry = feature.GetGeometryRef()
            if geometry:
                geom_wkt = geometry.ExportToWkt()
            else:
                geom_wkt = None

            data_rows.append({
                'fid': fid,
                traffic_column: traffic_value,
                'geometry_wkt': geom_wkt
            })

        # Clean up
        datasource = None

        if data_rows:
            df = pd.DataFrame(data_rows)
            print(f"  ✅ Successfully read {len(df)} features")
            print(f"  FID range: {df['fid'].min()} to {df['fid'].max()}")
            print(f"  FID unique count: {df['fid'].nunique()}")
            print(f"  Sample FIDs: {df['fid'].head().tolist()}")
            return df
        else:
            print(f"  ❌ No features read")
            return None

    except Exception as e:
        print(f"Error reading {filepath} with GDAL: {str(e)}")
        return None

def create_reference_geodataframe(reference_df):
    """
    Convert the reference DataFrame with WKT geometries to a GeoDataFrame
    """
    try:
        from shapely.wkt import loads

        # Convert WKT to Shapely geometries
        geometries = [loads(wkt) if wkt else None for wkt in reference_df['geometry_wkt']]

        # Create GeoDataFrame
        gdf = gpd.GeoDataFrame(
            reference_df[['fid']],
            geometry=geometries
        )

        # Remove any features with null geometry
        gdf = gdf[gdf.geometry.notna()]

        print(f"Created reference GeoDataFrame with {len(gdf)} features")
        return gdf

    except Exception as e:
        print(f"Error creating reference GeoDataFrame: {str(e)}")
        return None

def analyze_city_traffic_by_period_gdal(city_folder, traffic_column, output_folder):
    """
    Analyze traffic patterns using GDAL/OGR to read GPKG files
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get city name from folder name
    city = os.path.basename(city_folder).split('_')[-1]

    # Get list of all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))

    if not gpkg_files:
        print(f"No GPKG files found in {city_folder}")
        return

    print(f"Found {len(gpkg_files)} GPKG files")

    # Step 1: Read reference file to get geometry template
    reference_file = gpkg_files[0]
    print(f"\n{'='*60}")
    print(f"CREATING REFERENCE FROM: {os.path.basename(reference_file)}")
    print(f"{'='*60}")

    reference_df = read_gpkg_with_gdal(reference_file, traffic_column)
    if reference_df is None:
        return

    # Create reference GeoDataFrame for output
    reference_gdf = create_reference_geodataframe(reference_df)
    if reference_gdf is None:
        return

    # Step 2: Read all files using GDAL
    all_data = []
    print(f"\n{'='*60}")
    print(f"PROCESSING ALL FILES")
    print(f"{'='*60}")

    for i, file in enumerate(gpkg_files):
        df = read_gpkg_with_gdal(file, traffic_column)

        if df is not None:
            # Extract timestamp from filename
            timestamp = extract_timestamp_from_filename(file)
            if timestamp:
                df['timestamp'] = timestamp
                # Keep only the columns we need
                df_clean = df[['fid', traffic_column, 'timestamp']].copy()
                all_data.append(df_clean)

        # Progress indicator
        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(gpkg_files)} files...")

    if not all_data:
        print(f"No valid data found in {city_folder}")
        return

    # Step 3: Combine all data
    print(f"\n{'='*60}")
    print(f"COMBINING AND ANALYZING DATA")
    print(f"{'='*60}")

    combined_data = pd.concat(all_data, ignore_index=True)
    print(f"Combined data shape: {combined_data.shape}")

    # Add hour and time period
    combined_data['hour'] = combined_data['timestamp'].dt.hour
    combined_data['time_period'] = combined_data['hour'].apply(get_time_period)

    # Show detailed data summary
    print(f"\nData Summary:")
    print(f"  Total records: {len(combined_data)}")
    print(f"  Unique FIDs: {combined_data['fid'].nunique()}")
    print(f"  FID range: {combined_data['fid'].min()} to {combined_data['fid'].max()}")
    print(f"  Unique timestamps: {combined_data['timestamp'].nunique()}")
    print(f"  Timestamp range: {combined_data['timestamp'].min()} to {combined_data['timestamp'].max()}")

    # Show time period distribution
    print(f"\nTime period distribution:")
    period_counts = combined_data['time_period'].value_counts().sort_index()
    print(period_counts)

    # Step 4: Process each time period
    for period in combined_data['time_period'].unique():
        print(f"\n{'='*60}")
        print(f"PROCESSING TIME PERIOD: {period}")
        print(f"{'='*60}")

        # Filter data for this time period
        period_data = combined_data[combined_data['time_period'] == period]
        print(f"Records for {period}: {len(period_data)}")

        # Calculate statistics for each segment using FID
        period_stats = period_data.groupby('fid').agg({
            traffic_column: ['mean', 'std', 'count', 'min', 'max']
        }).round(2)

        # Show detailed debugging info
        print(f"\nDetailed segment analysis for {period}:")
        sample_fids = [1, 2, 3, 4, 5]  # Check first 5 FIDs

        for fid in sample_fids:
            if fid in period_data['fid'].values:
                sample_data = period_data[period_data['fid'] == fid]
                values = sample_data[traffic_column].tolist()
                print(f"FID {fid}: {len(values)} observations")
                print(f"  Values: {values[:10]}{'...' if len(values) > 10 else ''}")  # Show first 10 values
                print(f"  Mean: {sample_data[traffic_column].mean():.3f}")
                print(f"  Std: {sample_data[traffic_column].std():.3f}")

        # Check observations per segment
        obs_per_segment = period_data.groupby('fid').size()
        print(f"\nObservations per FID - Min: {obs_per_segment.min()}, Max: {obs_per_segment.max()}, Mean: {obs_per_segment.mean():.2f}")

        # Flatten column names
        period_stats.columns = [f"{traffic_column}_{col}" for col in ['mean', 'std', 'count', 'min', 'max']]
        period_stats = period_stats.reset_index()

        # Show sample statistics to verify variation
        print(f"\nFirst 10 FID statistics:")
        display_cols = ['fid', f'{traffic_column}_mean', f'{traffic_column}_std', f'{traffic_column}_count']
        print(period_stats.head(10)[display_cols])

        # Show variation in means to verify FIDs have different values
        means = period_stats[f'{traffic_column}_mean']
        print(f"\nVariation in FID means:")
        print(f"  Min mean: {means.min():.3f}")
        print(f"  Max mean: {means.max():.3f}")
        print(f"  Mean of means: {means.mean():.3f}")
        print(f"  Std of means: {means.std():.3f}")
        print(f"  Unique mean values: {means.nunique()}")

        # Merge with geometry from reference
        period_gdf = reference_gdf.merge(period_stats, on='fid', how='left')

        # Count how many FIDs have data
        valid_data_count = period_gdf[f'{traffic_column}_mean'].notna().sum()
        print(f"\nGeometry merge result:")
        print(f"  Total geometries: {len(period_gdf)}")
        print(f"  FIDs with data: {valid_data_count}")
        print(f"  FIDs without data: {len(period_gdf) - valid_data_count}")

        # Save as GPKG
        output_file = os.path.join(output_folder, f"{period}_{city}_gdal.gpkg")
        period_gdf.to_file(output_file, driver='GPKG')
        print(f"Saved {output_file}")

    print(f"\n{'='*60}")
    print("GDAL ANALYSIS COMPLETE!")
    print(f"{'='*60}")

# Example usage
if __name__ == "__main__":
    city_folder = "/content/traffic-data/traffic_data_smg"
    traffic_column = "jam_factor"
    output_folder = "/content/traffic-data"

    analyze_city_traffic_by_period_gdal(city_folder, traffic_column, output_folder)

No GPKG files found in /content/traffic-data/traffic_data_smg


In [ ]:
# GDAL with Spatial Matching for Consistent Segment IDs

import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime
from osgeo import ogr, osr
import geopandas as gpd
from shapely.wkt import loads
from shapely.geometry import Point
import hashlib

def extract_timestamp_from_filename(filename):
    """
    Extract timestamp from filename pattern: city_traffic_YYYYMMDD_HHMMSS.gpkg
    """
    try:
        # Get just the filename without path and remove extension
        base_name = os.path.splitext(os.path.basename(filename))[0]
        # Split by underscore and get date and time parts
        parts = base_name.split('_')
        date_str = parts[2]
        time_str = parts[3]
        # Combine date and time and parse
        datetime_str = f"{date_str}_{time_str}"
        return datetime.strptime(datetime_str, "%Y%m%d_%H%M%S")
    except Exception as e:
        print(f"Error parsing timestamp from {filename}: {str(e)}")
        return None

def get_time_period(hour):
    """
    Define time periods in a day
    """
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 9:
        return 'morning_peak'
    elif 9 <= hour < 12:
        return 'morning_offpeak'
    elif 12 <= hour < 14:
        return 'lunch_hours'
    elif 14 <= hour < 16:
        return 'afternoon_offpeak'
    elif 16 <= hour < 19:
        return 'evening_peak'
    elif 19 <= hour < 22:
        return 'evening_offpeak'
    else:
        return 'late_night'

def create_geometry_hash(geometry_wkt):
    """
    Create a hash from geometry WKT for spatial matching
    This helps identify the same geometry across different files
    """
    if geometry_wkt is None:
        return None

    # Normalize the WKT (remove extra spaces, etc.)
    normalized_wkt = ' '.join(geometry_wkt.split())

    # Create hash
    return hashlib.md5(normalized_wkt.encode()).hexdigest()

def read_gpkg_with_gdal_spatial(filepath, traffic_column):
    """
    Read GPKG file using GDAL/OGR and create spatial identifiers
    """
    try:
        # Open the GPKG file
        driver = ogr.GetDriverByName("GPKG")
        datasource = driver.Open(filepath, 0)  # 0 means read-only

        if datasource is None:
            print(f"Could not open {filepath}")
            return None

        # Get the layer (assuming first layer)
        layer = datasource.GetLayer(0)
        feature_count = layer.GetFeatureCount()

        # Get layer definition to see available fields
        layer_defn = layer.GetLayerDefn()
        field_names = []
        for i in range(layer_defn.GetFieldCount()):
            field_defn = layer_defn.GetFieldDefn(i)
            field_names.append(field_defn.GetName())

        # Check if traffic column exists
        if traffic_column not in field_names:
            print(f"  ❌ Traffic column '{traffic_column}' not found in {filepath}")
            return None

        # Read all features
        data_rows = []
        layer.ResetReading()  # Start from beginning

        for feature in layer:
            # Get original FID
            original_fid = feature.GetFID()

            # Get traffic value
            traffic_value = feature.GetField(traffic_column)

            # Get geometry as WKT
            geometry = feature.GetGeometryRef()
            if geometry:
                geom_wkt = geometry.ExportToWkt()
                geom_hash = create_geometry_hash(geom_wkt)

                # Get centroid for spatial matching
                centroid = geometry.Centroid()
                if centroid:
                    centroid_x = centroid.GetX()
                    centroid_y = centroid.GetY()
                else:
                    centroid_x = None
                    centroid_y = None
            else:
                geom_wkt = None
                geom_hash = None
                centroid_x = None
                centroid_y = None

            data_rows.append({
                'original_fid': original_fid,
                'geometry_hash': geom_hash,
                'centroid_x': centroid_x,
                'centroid_y': centroid_y,
                traffic_column: traffic_value,
                'geometry_wkt': geom_wkt
            })

        # Clean up
        datasource = None

        if data_rows:
            df = pd.DataFrame(data_rows)
            return df
        else:
            return None

    except Exception as e:
        print(f"Error reading {filepath} with GDAL: {str(e)}")
        return None

def create_master_segment_mapping(reference_df):
    """
    Create a master segment mapping based on spatial properties
    """
    print(f"Creating master segment mapping from {len(reference_df)} features")

    # Remove features without geometry
    valid_df = reference_df[reference_df['geometry_hash'].notna()].copy()

    # Assign consistent segment_id based on spatial properties
    # Sort by centroid coordinates to ensure consistency
    valid_df = valid_df.sort_values(['centroid_y', 'centroid_x']).reset_index(drop=True)
    valid_df['segment_id'] = range(1, len(valid_df) + 1)

    # Create mapping dictionary
    hash_to_segment = dict(zip(valid_df['geometry_hash'], valid_df['segment_id']))

    # Create reference GeoDataFrame for final output
    geometries = [loads(wkt) if wkt else None for wkt in valid_df['geometry_wkt']]
    reference_gdf = gpd.GeoDataFrame(
        valid_df[['segment_id']],
        geometry=geometries
    )

    print(f"Created master mapping with {len(hash_to_segment)} unique segments")
    print(f"Segment IDs range from 1 to {len(hash_to_segment)}")

    return hash_to_segment, reference_gdf

def assign_consistent_segment_ids(df, hash_to_segment_mapping):
    """
    Assign consistent segment IDs based on spatial matching
    """
    df = df.copy()

    # Map geometry hashes to segment IDs
    df['segment_id'] = df['geometry_hash'].map(hash_to_segment_mapping)

    # Count how many segments were successfully mapped
    mapped_count = df['segment_id'].notna().sum()
    total_count = len(df)

    print(f"  Spatial matching: {mapped_count}/{total_count} segments mapped ({mapped_count/total_count*100:.1f}%)")

    return df

def analyze_city_traffic_gdal_spatial(city_folder, traffic_column, output_folder):
    """
    Analyze traffic patterns using GDAL with spatial segment matching
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get city name from folder name
    city = os.path.basename(city_folder).split('_')[-1]

    # Get list of all GPKG files in the folder
    gpkg_files = glob.glob(os.path.join(city_folder, "*.gpkg"))

    if not gpkg_files:
        print(f"No GPKG files found in {city_folder}")
        return

    print(f"Found {len(gpkg_files)} GPKG files")

    # Step 1: Create master segment mapping from reference file
    reference_file = gpkg_files[0]
    print(f"\n{'='*70}")
    print(f"CREATING MASTER SEGMENT MAPPING FROM: {os.path.basename(reference_file)}")
    print(f"{'='*70}")

    reference_df = read_gpkg_with_gdal_spatial(reference_file, traffic_column)
    if reference_df is None:
        return

    hash_to_segment, reference_gdf = create_master_segment_mapping(reference_df)

    # Step 2: Process all files with spatial matching
    all_data = []
    print(f"\n{'='*70}")
    print(f"PROCESSING ALL FILES WITH SPATIAL MATCHING")
    print(f"{'='*70}")

    for i, file in enumerate(gpkg_files):
        df = read_gpkg_with_gdal_spatial(file, traffic_column)

        if df is not None:
            # Assign consistent segment IDs based on spatial matching
            df_with_ids = assign_consistent_segment_ids(df, hash_to_segment)

            # Extract timestamp from filename
            timestamp = extract_timestamp_from_filename(file)
            if timestamp:
                df_with_ids['timestamp'] = timestamp
                # Keep only successfully mapped segments
                df_clean = df_with_ids[df_with_ids['segment_id'].notna()][['segment_id', traffic_column, 'timestamp']].copy()
                all_data.append(df_clean)

        # Progress indicator
        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(gpkg_files)} files...")

    if not all_data:
        print(f"No valid data found in {city_folder}")
        return

    # Step 3: Combine and analyze data
    print(f"\n{'='*70}")
    print(f"COMBINING AND ANALYZING DATA")
    print(f"{'='*70}")

    combined_data = pd.concat(all_data, ignore_index=True)
    print(f"Combined data shape: {combined_data.shape}")

    # Add hour and time period
    combined_data['hour'] = combined_data['timestamp'].dt.hour
    combined_data['time_period'] = combined_data['hour'].apply(get_time_period)

    # Show detailed data summary
    print(f"\nData Summary:")
    print(f"  Total records: {len(combined_data)}")
    print(f"  Unique segment IDs: {combined_data['segment_id'].nunique()}")
    print(f"  Segment ID range: {combined_data['segment_id'].min()} to {combined_data['segment_id'].max()}")
    print(f"  Unique timestamps: {combined_data['timestamp'].nunique()}")
    print(f"  Timestamp range: {combined_data['timestamp'].min()} to {combined_data['timestamp'].max()}")

    # Show time period distribution
    print(f"\nTime period distribution:")
    period_counts = combined_data['time_period'].value_counts().sort_index()
    print(period_counts)

    # Step 4: Process each time period
    for period in combined_data['time_period'].unique():
        print(f"\n{'='*70}")
        print(f"PROCESSING TIME PERIOD: {period}")
        print(f"{'='*70}")

        # Filter data for this time period
        period_data = combined_data[combined_data['time_period'] == period]
        print(f"Records for {period}: {len(period_data)}")

        # Calculate statistics for each segment
        period_stats = period_data.groupby('segment_id').agg({
            traffic_column: ['mean', 'std', 'count', 'min', 'max']
        }).round(3)

        # Show detailed debugging info
        print(f"\nDetailed segment analysis for {period}:")
        sample_segments = [1, 2, 3, 4, 5]  # Check first 5 segments

        for seg_id in sample_segments:
            if seg_id in period_data['segment_id'].values:
                sample_data = period_data[period_data['segment_id'] == seg_id]
                values = sample_data[traffic_column].tolist()
                print(f"Segment {seg_id}: {len(values)} observations")
                print(f"  Values: {values[:8]}{'...' if len(values) > 8 else ''}")
                print(f"  Mean: {sample_data[traffic_column].mean():.3f}")
                print(f"  Std: {sample_data[traffic_column].std():.3f}")

        # Check observations per segment
        obs_per_segment = period_data.groupby('segment_id').size()
        print(f"\nObservations per segment - Min: {obs_per_segment.min()}, Max: {obs_per_segment.max()}, Mean: {obs_per_segment.mean():.1f}")

        # Flatten column names
        period_stats.columns = [f"{traffic_column}_{col}" for col in ['mean', 'std', 'count', 'min', 'max']]
        period_stats = period_stats.reset_index()

        # Show sample statistics to verify variation
        print(f"\nFirst 10 segment statistics:")
        display_cols = ['segment_id', f'{traffic_column}_mean', f'{traffic_column}_std', f'{traffic_column}_count']
        print(period_stats.head(10)[display_cols])

        # Show variation in means to verify segments have different values
        means = period_stats[f'{traffic_column}_mean']
        print(f"\nVariation in segment means:")
        print(f"  Min mean: {means.min():.3f}")
        print(f"  Max mean: {means.max():.3f}")
        print(f"  Mean of means: {means.mean():.3f}")
        print(f"  Std of means: {means.std():.3f}")
        print(f"  Unique mean values: {means.nunique()}")

        if means.std() > 0.001:  # Check if there's meaningful variation
            print(f"  ✅ Good variation in segment means!")
        else:
            print(f"  ⚠️  Low variation in segment means - might indicate data issues")

        # Merge with geometry
        period_gdf = reference_gdf.merge(period_stats, on='segment_id', how='left')

        # Count how many segments have data
        valid_data_count = period_gdf[f'{traffic_column}_mean'].notna().sum()
        print(f"\nGeometry merge result:")
        print(f"  Total segments: {len(period_gdf)}")
        print(f"  Segments with data: {valid_data_count}")
        print(f"  Segments without data: {len(period_gdf) - valid_data_count}")

        # Save as GPKG
        output_file = os.path.join(output_folder, f"{period}_{city}_spatial.gpkg")
        period_gdf.to_file(output_file, driver='GPKG')
        print(f"Saved {output_file}")

    print(f"\n{'='*70}")
    print("SPATIAL MATCHING ANALYSIS COMPLETE!")
    print(f"{'='*70}")

# Example usage
if __name__ == "__main__":
    city_folder = "/content/traffic-data/traffic_data_smg"
    traffic_column = "jam_factor"
    output_folder = "/content/traffic-data"

    analyze_city_traffic_gdal_spatial(city_folder, traffic_column, output_folder)